# 04 特徴量エンジニアリング(causal)

`irp.features` は **先読みゼロ(causal)** の特徴量を提供する:`feature[t]` は `t` 以前の情報だけに依存する(rolling は trailing、`pct_change` は前埋めしない、横断変換は同日の cross-section のみ)。本 NB は合成データで主要特徴量を計算し、**因果性(未来不変性)** を数値で実証し、最後に **point-in-time マクロ特徴量** を示す。

In [ ]:
import numpy as np
import pandas as pd

# Deterministic synthetic price panel (no network needed; the same APIs work
# on real data via irp.data.get_prices / price_panel). Five assets with
# different drift and volatility so feature/signal behavior is predictable.
rng = np.random.default_rng(7)
idx = pd.bdate_range('2018-01-01', periods=750)
drift = {'AAA': 0.0002, 'BBB': 0.0004, 'CCC': 0.0006, 'DDD': 0.0008, 'EEE': 0.0010}
vol = {'AAA': 0.008, 'BBB': 0.010, 'CCC': 0.012, 'DDD': 0.018, 'EEE': 0.025}
steps = {a: rng.normal(drift[a], vol[a], len(idx)) for a in drift}
prices = pd.DataFrame({a: 100 * np.exp(np.cumsum(s)) for a, s in steps.items()}, index=idx)
prices.tail(3)

## 価格特徴量(リターン・モメンタム・実現ボラ・z-score・ドローダウン)

In [ ]:
from irp import features as F

feat = pd.DataFrame({
    'ret_1d': F.returns(prices['EEE']),
    'mom_12_1': F.momentum(prices['EEE'], lookback=252, skip=21),
    'rvol_21': F.realized_volatility(prices['EEE'], 21),
    'z_60': F.rolling_zscore(prices['EEE'], 60),
    'drawdown': F.drawdown(prices['EEE']),
})
feat.tail(4)

## 横断特徴量(同日のアセット間 z-score / rank)

各日付の行だけで標準化するので、時間軸の先読みは入らない。

In [ ]:
mom = F.momentum(prices, lookback=252, skip=21)
z_xs = F.cross_sectional_zscore(mom)
rank_xs = F.cross_sectional_rank(mom)
print('cross-sectional z (last date):')
display(z_xs.tail(1).T)
print('cross-sectional rank (last date):')
display(rank_xs.tail(1).T)

## 因果性の実証:未来を切り落としても過去の特徴量は変わらない

プラットフォームの核となる不変条件。`feature(prices[:k])` は `feature(prices)[:k]` と **完全一致** する(未来を覗いていれば一致しない)。

In [ ]:
k = 500
full = F.momentum(prices, lookback=252, skip=21)
trunc = F.momentum(prices.iloc[:k], lookback=252, skip=21)
identical = full.iloc[:k].equals(trunc)
print('momentum(prices[:k]) == momentum(prices)[:k] :', identical)
assert identical, 'look-ahead detected!'
# same for a rolling z-score
fz, tz = F.rolling_zscore(prices, 60), F.rolling_zscore(prices.iloc[:k], 60)
print('rolling_zscore causal :', fz.iloc[:k].equals(tz))

## 前埋めしない(no silent forward-fill)

価格に欠損があると、リターンは前埋めで埋めず **欠損のまま** にする。

In [ ]:
gapped = prices['EEE'].copy()
gapped.iloc[100] = np.nan  # inject a gap
r = F.returns(gapped)
print('return at the bar AFTER the gap is NaN (not fabricated):',
      bool(np.isnan(r.iloc[101])))

## point-in-time マクロ特徴量

マクロは `as_of` 経由で評価するので **release_date 前は使えない**。`pit_feature_frame` は当時見えた値・対象期間・公表日・経過日数(staleness)を返す。

In [ ]:
from irp.macro.schema import MacroObservation, to_macro_frame
obs = [
    MacroObservation('cpi','US',pd.Timestamp('2024-01-01'),pd.Timestamp('2024-01-31'),
                     pd.Timestamp('2024-02-15'),3.0,'ALFRED',vintage_available=True),
    MacroObservation('cpi','US',pd.Timestamp('2024-01-01'),pd.Timestamp('2024-01-31'),
                     pd.Timestamp('2024-03-15'),3.2,'ALFRED',vintage_available=True),
    MacroObservation('cpi','US',pd.Timestamp('2024-02-01'),pd.Timestamp('2024-02-29'),
                     pd.Timestamp('2024-03-15'),2.8,'ALFRED',vintage_available=True),
]
mf = to_macro_frame(obs)
pit = F.pit_feature_frame(mf, ['2024-02-20', '2024-03-20'])
display(pit)
print('YoY-style change at 2024-03-20:', round(F.pit_change(mf, ['2024-03-20']).iloc[0], 4))

2024-02-20 時点では 1月分の **初出 3.0** だけが見える(2月分は未公表、1月の改定は未来)。これが改定マクロでの未来漏れを防ぐ。特徴量層は全てこの causal 性の上に成り立つ。